In [27]:
import os

In [28]:
OPENAI_API_KEY="REMOVED_OPENAI_API_KEY"

In [29]:
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

In [31]:
from flow import execute

In [32]:
chunks = execute()

[INFO] Starting RFP processing pipeline.
[INFO] Extracting text from PDF: /home/kparth/HomersHackers/parth/data/ELIGIBLE_RFP_2.pdf
[INFO] Extraction complete.
[INFO] Cleaning extracted text.
[INFO] Cleaning complete.
[INFO] Chunking the cleaned document.


In [33]:
chunks

['Hazelwood School District Request for Proposals RFP for Information Technology Audit Services Due : February 25 , 2025 Time : 10 : 00 a . m . CDT February 3 , 2025 REQUEST FOR PROPOSAL INFORMATION TECHNOLOGY AUDITING SERVICES The Hazelwood School District seeks proposals from experienced firms to conduct a comprehensive information technology audit for the Hazelwood School District HSD . This audit will collect and evaluate evidence of HSDs information technology systems , practices , and operations to determine if changes are needed in the existing structure to meet current and future needs . Any questions regarding the specifications are to be directed to Danielle Thomas , Director of Purchasing Supplier Diversity no later than 2 : 00 pm on Tuesday , February 18 , 2025 through the Vendor Registry online question submission process via the districts website at https : www . hazelwoodschools . orgPage2238 . Only these inquiries will be answered . Any items requiring clarification wil

In [34]:
print(len(chunks))

19


In [35]:
import re
from sentence_transformers import SentenceTransformer, util

def clean_text(text):
    return text.strip()

chunks = [clean_text(chunk) for chunk in chunks]
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks, convert_to_tensor=True)
reference_clauses = [
    "The Contractor may offer additional maintenance services if eligible.",
    "Future support services may enhance bid eligibility.",
    "Optional maintenance support may be provided to enhance bid competitiveness."
]
ref_embeddings = model.encode(reference_clauses, convert_to_tensor=True)
threshold = 0.4
matched_chunks = []

for chunk, embedding in zip(chunks, chunk_embeddings):
    cosine_scores = util.cos_sim(embedding, ref_embeddings)
    max_score = cosine_scores.max().item()
    if max_score >= threshold:
        matched_chunks.append((chunk, max_score))
print("Matched Chunks:")
for chunk, score in matched_chunks:
    print(f"Chunk: \"{chunk}\" | Similarity Score: {score:.2f}")


Matched Chunks:
Chunk: "due to the reasons cited below : Attach additional pages as necessary Firm Name and Contact Person Mailing Address Reason not utilized 21 HSD FORM B 22 Additional Information E - Verify Section 285 . 530 , RSMo , requires businesses that contract with districts for services that may exceed 5 , 000 to provide affidavits affirming that the businesses use E - Verify and do not employ illegal workers in connection with the contract . OSHA Training 292 . 675 , RSMo . , requires contractors and subcontractors to provide a ten 10 hour OSHA construction safety program for on - site workers of public works projects . If employeesworkerslaborers of the contractor or subcontractor have already completed the training program , they must have documentation of completing the program . They do not need to retake this training . Client References Contractors must supply at least three 3 client references for completed work in the last 1 - 2years . Please provide the organizatio

In [36]:
from langchain_openai import OpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


prompt_template = """
You are a proposal expert. Here is a contract clause that was identified as beneficial:

"{clause}"

Summarize the clause in simple language and explain what actions I can take to improve my proposal based on this clause.
Provide clear, short, and actionable advice.
"""

prompt = PromptTemplate(
    input_variables=["clause"],
    template=prompt_template
)

chain = prompt | llm | StrOutputParser()

print("Matched Chunks and Improvement Advice:")
for clause, score in matched_chunks:
    response = chain.invoke({"clause": clause})
    print(f"Chunk: \"{clause}\" | Similarity Score: {score:.2f}")
    print("Improvement Advice:")
    print(response)
    print("-" * 80)


Matched Chunks and Improvement Advice:
Chunk: "due to the reasons cited below : Attach additional pages as necessary Firm Name and Contact Person Mailing Address Reason not utilized 21 HSD FORM B 22 Additional Information E - Verify Section 285 . 530 , RSMo , requires businesses that contract with districts for services that may exceed 5 , 000 to provide affidavits affirming that the businesses use E - Verify and do not employ illegal workers in connection with the contract . OSHA Training 292 . 675 , RSMo . , requires contractors and subcontractors to provide a ten 10 hour OSHA construction safety program for on - site workers of public works projects . If employeesworkerslaborers of the contractor or subcontractor have already completed the training program , they must have documentation of completing the program . They do not need to retake this training . Client References Contractors must supply at least three 3 client references for completed work in the last 1 - 2years . Please 